# 03 · Feature prototype (3 example places)

The features the search engine will use, run on three places so we can eyeball them:

| Feature | Tool | Used for |
|---|---|---|
| Sentences + lemmas | spaCy `en_core_web_sm` | keyword (BM25) index |
| Sentence embeddings | `all-mpnet-base-v2` | meaning-based search |
| Top 5 vibe phrases | KeyBERT (same mpnet model) | place summaries |
| Sentence sentiment | `cardiffnlp/twitter-roberta-base-sentiment-latest` | "cozy" vs. "not cozy at all" |

First run downloads about 1 GB of models into `~/.cache/huggingface`.

In [ ]:
import json

import pandas as pd
import spacy
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from transformers import pipeline

from vibecheck.filter_sf import PLACES_OUT, REVIEWS_OUT
from vibecheck.report import write_section

places = pd.read_parquet(PLACES_OUT)
reviews = pd.read_parquet(REVIEWS_OUT)
places["atmosphere"] = places["MISC"].map(lambda s: json.loads(s).get("Atmosphere", []) if s else [])

## Pick 3 places

The most-reviewed cafe tagged *Cozy*, bar tagged *Trendy* and restaurant tagged *Upscale*,
so the vibe phrases can be compared with Google's tags.

In [ ]:
def pick(group: str, tag: str) -> pd.Series:
    """Most-reviewed place in `group` with Atmosphere `tag` (or without the tag if none has it)."""
    pool = places[places["group"] == group]
    tagged = pool[pool["atmosphere"].map(lambda tags: tag in tags)]
    return (tagged if len(tagged) else pool).sort_values("num_of_reviews", ascending=False).iloc[0]

examples = pd.DataFrame([pick("cafe", "Cozy"), pick("bar", "Trendy"), pick("restaurant", "Upscale")])
examples[["name", "group", "atmosphere", "avg_rating", "num_of_reviews"]]

## 1. spaCy: sentences and lemmas (BM25 keywords)

In [ ]:
nlp = spacy.load("en_core_web_sm", exclude=["ner"])

def keywords(sent) -> list[str]:
    """Lowercased lemmas of content words: what BM25 will index."""
    return [t.lemma_.lower() for t in sent if t.is_alpha and not t.is_stop]

rows = []
for place in examples.itertuples():
    texts = reviews.loc[reviews["gmap_id"] == place.gmap_id, "text"]
    for doc in nlp.pipe(texts):
        rows += [{"gmap_id": place.gmap_id, "name": place.name, "sentence": s.text.strip(), "lemmas": keywords(s)}
                 for s in doc.sents if s.text.strip()]
sentences = pd.DataFrame(rows)
print(f"{len(sentences):,} sentences from {len(examples)} places")
sentences.head(5)

## 2. Sentence embeddings (all-mpnet-base-v2)

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embeddings = encoder.encode(sentences["sentence"].tolist(), batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True)
print("embeddings shape:", embeddings.shape)  # (sentences, 768)

## 3. KeyBERT: top 5 vibe phrases per place

Candidates are 1-3 word phrases from all of a place's reviews. MMR (`diversity=0.5`) stops the
five phrases from being near-duplicates.

In [ ]:
kw_model = KeyBERT(model=encoder)

def vibe_phrases(texts: pd.Series, n: int = 5) -> list[str]:
    """Top `n` diverse key phrases for one place's reviews."""
    found = kw_model.extract_keywords(" ".join(texts), keyphrase_ngram_range=(1, 3), stop_words="english",
                                      use_mmr=True, diversity=0.5, top_n=n)
    return [phrase for phrase, _ in found]

phrases = {p.gmap_id: vibe_phrases(reviews.loc[reviews["gmap_id"] == p.gmap_id, "text"]) for p in examples.itertuples()}
phrases

## 4. Sentence sentiment (twitter-roberta)

In [ ]:
sentiment = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                     truncation=True, max_length=128)
sentences["sentiment"] = [r["label"] for r in sentiment(sentences["sentence"].tolist(), batch_size=32)]
sentences[["name", "sentence", "sentiment"]].sample(8, random_state=0)

## 5. Summary table for the slides

In [ ]:
shares = sentences.groupby("gmap_id")["sentiment"].value_counts(normalize=True).unstack(fill_value=0)
summary = pd.DataFrame(
    {
        "place": examples["name"].values,
        "Google Atmosphere": examples["atmosphere"].map(", ".join).values,
        "sentences": sentences.groupby("gmap_id").size().reindex(examples["gmap_id"]).values,
        "top 5 vibe phrases (KeyBERT)": [", ".join(phrases[g]) for g in examples["gmap_id"]],
        "% positive": (100 * shares.reindex(examples["gmap_id"])["positive"]).round(0).values,
        "% negative": (100 * shares.reindex(examples["gmap_id"])["negative"]).round(0).values,
    }
)
md = f"""## Features prototype (notebook 03)

Embeddings: {embeddings.shape[0]:,} sentences x {embeddings.shape[1]} dims (all-mpnet-base-v2).

{summary.to_markdown(index=False)}
"""
write_section("features", md)
summary